#### Starting simple Finetuning experiments

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from pathlib import Path
import requests
from dotenv import load_dotenv

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

In [4]:
import torch
import torch.nn as nn 
from torch.nn import functional as F
import tiktoken

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
device

In [5]:
load_dotenv( Path.cwd() / "env.env")
PATH_TO_DATA_DIR = Path.cwd() / "dataset"
PATH_TO_DATA_DIR

### Step-1:- Loading Dataset (Spam)

In [6]:
from model_training.post_training.artifacts.data.finetuning.spam_loader import load_dataset, create_balanced_dataset, random_split

In [7]:
df = load_dataset(PATH_TO_DATA_DIR)
NUM_CLASSES = len(set(df['Label']))
print(df["Label"].value_counts())

In [8]:
balanced_df = create_balanced_dataset(df)
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})    
balanced_df.shape

In [9]:
train_df, val_df, test_df = random_split(balanced_df, 0.7, 0.1)
print(train_df.columns)
train_df.shape,val_df.shape, test_df.shape

In [10]:
print(train_df.shape, val_df.shape)
plt.figure(figsize =(20, 7))
sns.countplot(x="Label", data= df)
# plt.xticks(rotation=90, ha="right")
plt.grid()
plt.show()

### Basic Finetuning

In [11]:
from model_training.framework.data.post_train.finetuning.classification_dataset import CustomDataLoader, return_dataloader

In [12]:
tokenizer = tiktoken.get_encoding("gpt2")
pad_token_id = (tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))[0]

In [13]:
train_dataset = CustomDataLoader(df = train_df, 
                                train_column ="Text", 
                                label_column = "Label",
                                tokenizer= tokenizer, pad_token_id = pad_token_id)
max_length = train_dataset.max_length
max_length

In [14]:
val_dataset = CustomDataLoader(df = val_df, train_column ="Text", label_column = "Label",
                            tokenizer= tokenizer, pad_token_id = pad_token_id, max_length=max_length)
test_dataset = CustomDataLoader(df = test_df, train_column ="Text", label_column = "Label",
                            tokenizer= tokenizer, pad_token_id = pad_token_id, max_length=max_length)

In [15]:
num_workers = 0
batch_size = 8
torch.manual_seed(123)

In [16]:
train_loader = return_dataloader(dataset=train_dataset, batch_size=batch_size, shuffle=True, workers=num_workers, drop_last=True)
val_loader = return_dataloader(dataset=val_dataset, batch_size=batch_size, shuffle=True, workers=num_workers, drop_last=True)
test_loader = return_dataloader(dataset=test_dataset, batch_size=batch_size, shuffle=True, workers=num_workers, drop_last=True)

In [17]:
for input_batch, target_batch in train_loader:
    pass
print("Input batch dimensions:", input_batch.shape)
print("Label batch dimensions", target_batch.shape)

In [18]:
print(f"{len(train_loader)} training batches")
print(f"{len(val_loader)} validation batches")
print(f"{len(test_loader)} test batches")

### Step-2: - Loading the Model 

In [19]:
from model_training.framework.model.load_model.gpt2.configs import load_model_config

In [20]:
config = load_model_config(train_dataset.max_length, "gpt2-small")

In [21]:
from model_training.framework.model.load_model.gpt2.model import GPT

In [22]:
model = GPT.from_pretrained(config, "gpt2")
print(f"Model Loaded: {model}")
model.eval()
model = model.to(device)

### Testing the model

In [23]:
print(config.input_prompt)
enc_input = tokenizer.encode(config.input_prompt)
enc_input = torch.tensor(enc_input, dtype = torch.long).to(device)
enc_input = enc_input.unsqueeze(0)
print(enc_input, (enc_input).shape)

In [24]:
from model_training.framework.model.components.inference.inference import generate_prob, gen_sample

In [25]:
decoded_text = gen_sample(model, enc_input, max_new_tokens=40, context_len=config.context_len)
print(tokenizer.decode(decoded_text.squeeze(0).tolist()))

#### Finetuning Model Changes -> Depending on dataset

In [26]:
model

In [27]:
torch.manual_seed(123)

In [ ]:
### Setting Last 3-4 layers as trainable
# for i in model.transformer.h[:-LAST_X_LAYERS]:
#     i.requires_grad = True


In [30]:
LAST_X_LAYERS = 2
for param in model.transformer.h[-LAST_X_LAYERS:].parameters():
    # print("here->", i)
    param.requires_grad=True
for param in model.transformer.ln_f.parameters():
    param.requires_grad = True

In [31]:
## print all Trainable layer
for name, param in model.named_parameters():
    if param.requires_grad == True:
        print(name, param.requires_grad, end = '|           ')


In [32]:
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(trainable_count)

In [33]:
## After Changing Last Layer -> Only 2 classes 
inputs = tokenizer.encode("Do you have time")
inputs = torch.tensor(inputs).unsqueeze(0)
inputs = inputs.to(device)
model = model.to(device)
print("Inputs:", inputs)
print("Inputs dimensions:", inputs.shape) # shape: (batch_size, num_tokens)

with torch.no_grad():
    loss, outputs = model(inputs)

print("Outputs:\n", outputs)
print("Outputs dimensions:", outputs.shape) # shape: (batch_size, num_tokens, num_classes)

### Now Setup Loss Functions 

In [34]:
### Sample Work 
logits = outputs[:, -1, :]
probs = F.softmax(logits, dim=-1)
label = torch.argmax(probs).item()
print(probs, label)

In [35]:
def calculate_loss(logits, target, device):
    target = target.to(device)
    pred = logits[:, -1, :]
    loss = F.cross_entropy(pred, target)
    return loss


def basic_pred(input_data, model, device):
    # target = target.to(device)
    input_data= input_data.to(device)
    # print(input_batch.shape, target_batch.shape)
    model = model.to(device)
    _, logits = model(idx = input_data)
    return logits
    

def calc_loss_loader(dataloader, model, device, num_batches=None):
    total_loss = 0.
    if len(dataloader) ==0:
        raise "No Data in dataloader"
    else:
        num_batches = min(num_batches, len(dataloader))
        for i, (input_batch, target_batch) in enumerate(dataloader):
            
            if i<num_batches:
                logits = basic_pred(input_batch, model, device)
                logits= logits.to(device)
                loss = calculate_loss(logits, target_batch, device)
                total_loss+=loss.item()
            else:
                break
    return total_loss/num_batches

In [36]:
with torch.no_grad(): # Disable gradient tracking for efficiency because we are not training, yet
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
    test_loss = calc_loss_loader(test_loader, model, device, num_batches=5)

print(f"Training loss: {train_loss:.3f}")
print(f"Validation loss: {val_loss:.3f}")
print(f"Test loss: {test_loss:.3f}")

In [37]:
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct_predictions, num_examples = 0, 0

    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch, target_batch = input_batch.to(device), target_batch.to(device)

            with torch.no_grad():
                _ , logits = model(input_batch)  # Logits of last output token
            predicted_labels = torch.argmax(logits[:, -1, :], dim=-1)

            num_examples += predicted_labels.shape[0]
            correct_predictions += (predicted_labels == target_batch).sum().item()
        else:
            break
    return correct_predictions / num_examples

In [38]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    _ ,logits = model(input_batch)
    # Logits of last output token
    loss = torch.nn.functional.cross_entropy(logits[:, -1, :], target_batch)
    return loss

# Overall the same as `train_model_simple` in chapter 5
def train_classifier_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                            eval_freq, eval_iter):
    # Initialize lists to track losses and examples seen
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    examples_seen, global_step = 0, -1

    # Main training loop
    for epoch in range(num_epochs):
        model.train()  # Set model to training mode

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad() # Reset loss gradients from previous batch iteration
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward() # Calculate loss gradients
            optimizer.step() # Update model weights using loss gradients
            examples_seen += input_batch.shape[0] # New: track examples instead of tokens
            global_step += 1

            # Optional evaluation step
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # Calculate accuracy after each epoch
        train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=eval_iter)
        val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=eval_iter)
        print(f"Training accuracy: {train_accuracy*100:.2f}% | ", end="")
        print(f"Validation accuracy: {val_accuracy*100:.2f}%")
        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)

    return train_losses, val_losses, train_accs, val_accs, examples_seen

In [39]:
# Same as chapter 5
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

In [40]:
import time

start_time = time.time()

torch.manual_seed(123)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)

num_epochs = 5
train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=50, eval_iter=5,
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

In [41]:
import matplotlib.pyplot as plt

def plot_values(epochs_seen, examples_seen, train_values, val_values, label="loss"):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    # Plot training and validation loss against epochs
    ax1.plot(epochs_seen, train_values, label=f"Training {label}")
    ax1.plot(epochs_seen, val_values, linestyle="-.", label=f"Validation {label}")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel(label.capitalize())
    ax1.legend()

    # Create a second x-axis for examples seen
    ax2 = ax1.twiny()  # Create a second x-axis that shares the same y-axis
    ax2.plot(examples_seen, train_values, alpha=0)  # Invisible plot for aligning ticks
    ax2.set_xlabel("Examples seen")

    fig.tight_layout()  # Adjust layout to make room
    plt.savefig(f"{label}-plot.pdf")
    plt.show()

In [42]:
epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_losses))

plot_values(epochs_tensor, examples_seen_tensor, train_losses, val_losses)

In [43]:
epochs_tensor = torch.linspace(0, num_epochs, len(train_accs))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_accs))

plot_values(epochs_tensor, examples_seen_tensor, train_accs, val_accs, label="accuracy")

In [44]:
train_accuracy = calc_accuracy_loader(train_loader, model, device)
val_accuracy = calc_accuracy_loader(val_loader, model, device)
test_accuracy = calc_accuracy_loader(test_loader, model, device)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

In [45]:
model.transformer

In [46]:
def classify_review(text, model, tokenizer, device, max_length=None, pad_token_id=50256):
    model.eval()

    # Prepare inputs to the model
    input_ids = tokenizer.encode(text)
    supported_context_length = model.transformer.wpe.weight.shape[0]
    # Note: In the book, this was originally written as pos_emb.weight.shape[1] by mistake
    # It didn't break the code but would have caused unnecessary truncation (to 768 instead of 1024)

    # Truncate sequences if they too long
    input_ids = input_ids[:min(max_length, supported_context_length)]
    assert max_length is not None, (
        "max_length must be specified. If you want to use the full model context, "
        "pass max_length=model.pos_emb.weight.shape[0]."
    )
    assert max_length <= supported_context_length, (
        f"max_length ({max_length}) exceeds model's supported context length ({supported_context_length})."
    )    
    # Alternatively, a more robust version is the following one, which handles the max_length=None case better
    # max_len = min(max_length,supported_context_length) if max_length else supported_context_length
    # input_ids = input_ids[:max_len]
    
    # Pad sequences to the longest sequence
    input_ids += [pad_token_id] * (max_length - len(input_ids))
    input_tensor = torch.tensor(input_ids, device=device).unsqueeze(0) # add batch dimension

    # Model inference
    with torch.no_grad():
        _, logits = model(input_tensor)  # Logits of the last output token
    predicted_label = torch.argmax(logits[:, -1, :], dim=-1).item()

    # Return the classified result
    return "spam" if predicted_label == 1 else "not spam"

In [47]:
text_1 = (
    "You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award."
)

print(classify_review( text_1, model, tokenizer, device, max_length=train_dataset.max_length
))